# Scrape healthcare.by

In [5]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [29]:
import re

In [6]:
url = "https://healthcare.by/search.php?typ=707&sort=1"

In [41]:
fap = "https://healthcare.by/search.php?typ=707&sort=1&page=" 
amb = "https://healthcare.by/search.php?typ=600&sort=1&page=" 
hospital = "https://healthcare.by/search.php?typ=603&sort=1&page="
polyclinic = "https://healthcare.by/search.php?typ=598&sort=1&page="

In [42]:
links = [fap, amb, hospital, polyclinic]

In [9]:
response = requests.get(url)
doc = BeautifulSoup(response.text)

In [10]:
doc

<html>
<head>
<meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
<meta content="ru" http-equiv="Content-Language"/>
<title>Здравоохранение Беларуси — результаты поиска</title>
<meta content="noindex" name="robots"/>
<meta content="" name="description"/>
<meta content="здравоохранение,беларусь,каталог,медицина,поиск" name="keywords"/>
<link href="styles/main.css" rel="stylesheet"/>
<script src="scripts/main.js"></script>
</head>
<body bgcolor="white">
<!-- Yandex.Metrika counter -->
<div style="display:none;"><script type="text/javascript">
    (function(w, c) {
        (w[c] = w[c] || []).push(function() {
            try {
                w.yaCounter10063810 = new Ya.Metrika({id:10063810, enableAll: true});
            }
            catch(e) { }
        });
    })(window, "yandex_metrika_callbacks");
    </script></div>
<script defer="defer" src="//mc.yandex.ru/metrika/watch.js" type="text/javascript"></script>
<noscript><div><img alt="" src="//mc.yandex.ru/watch/100

In [11]:
items = doc.select("li")

In [12]:
items

[<li style="font-size: 10pt"><a href="instinfo.php?orgnum=16445">Агеевский ФАП</a> (Агеевка)</li>,
 <li style="font-size: 10pt"><a href="instinfo.php?orgnum=17185">Адамовский ФАП</a> (Адамово)</li>,
 <li style="font-size: 10pt"><a href="instinfo.php?orgnum=18082">Адаховщинский ФАП</a> (Адаховщина)</li>,
 <li style="font-size: 10pt"><a href="instinfo.php?orgnum=15701">Азделинский ФАП</a> (Азделино)</li>,
 <li style="font-size: 10pt"><a href="instinfo.php?orgnum=15299">Азинский ФАП</a> (Азино)</li>,
 <li style="font-size: 10pt"><a href="instinfo.php?orgnum=16081">Александровский ФАП</a> (Александровка)</li>,
 <li style="font-size: 10pt"><a href="instinfo.php?orgnum=15850">Александровский ФАП</a> (Александровка)</li>,
 <li style="font-size: 10pt"><a href="instinfo.php?orgnum=17747">Алексеевичский ФАП</a> (Алексеевичи)</li>,
 <li style="font-size: 10pt"><a href="instinfo.php?orgnum=15416">Алексиничский ФАП</a> (Алексиничи)</li>,
 <li style="font-size: 10pt"><a href="instinfo.php?orgnum=169

In [21]:
items[0].find("a").next_sibling.strip(" \xa0()")

'Агеевка'

In [23]:
for item in items:
    print("----")
    print(item.select_one("a").get_text()) #name
    print(f"https://healthcare.by/{item.select_one("a")['href']}") #link
    print(item.find("a").next_sibling.strip(" \xa0()")) #town

----
Агеевский ФАП
https://healthcare.by/instinfo.php?orgnum=16445
Агеевка
----
Адамовский ФАП
https://healthcare.by/instinfo.php?orgnum=17185
Адамово
----
Адаховщинский ФАП
https://healthcare.by/instinfo.php?orgnum=18082
Адаховщина
----
Азделинский ФАП
https://healthcare.by/instinfo.php?orgnum=15701
Азделино
----
Азинский ФАП
https://healthcare.by/instinfo.php?orgnum=15299
Азино
----
Александровский ФАП
https://healthcare.by/instinfo.php?orgnum=16081
Александровка
----
Александровский ФАП
https://healthcare.by/instinfo.php?orgnum=15850
Александровка
----
Алексеевичский ФАП
https://healthcare.by/instinfo.php?orgnum=17747
Алексеевичи
----
Алексиничский ФАП
https://healthcare.by/instinfo.php?orgnum=15416
Алексиничи
----
Алешковичский ФАП
https://healthcare.by/instinfo.php?orgnum=16950
Олешковичи
----
Альбянский ФАП
https://healthcare.by/instinfo.php?orgnum=14945
Биозавода
----
Амбросовичский ФАП
https://healthcare.by/instinfo.php?orgnum=15506
Амбросовичи
----
Амховский ФАП, ФАП
https://h

In [24]:
def get_facility_type(name):
    name = name.lower().replace("ё", "е")

    facility_types = {
        "ФАП": ("FAP", ("фап", "фельдшерско-акушерский пункт")),
        "Амбулатория": ("Outpatient Clinic", ("амбулатория", "АВОП", "авоп", "ВАП", "вап")),
        "Больница": ("Hospital", ("больница",)),
        "Поликлиника": ("Polyclinic", ("поликлиника",)),
    }

    for facility_type, (type_eng, keywords) in facility_types.items():
        if any(keyword in name for keyword in keywords):
            return facility_type, type_eng

    return None, None

In [25]:
#Building a list of dictionaries

rows = []

for item in items:
    name = item.select_one("a").get_text(" ", strip=True)
    facility_type, type_eng = get_facility_type(name)
    row = {
        'facility_title': item.select_one("a").get_text(),
        'link': f"https://healthcare.by/{item.select_one("a")['href']}",
        'settlement': item.find("a").next_sibling.strip(" \xa0()"),
        'facility_type': facility_type,
        'facility_type_eng': type_eng
    }
    rows.append(row)
rows

[{'facility_title': 'Агеевский ФАП',
  'link': 'https://healthcare.by/instinfo.php?orgnum=16445',
  'settlement': 'Агеевка',
  'facility_type': 'ФАП',
  'facility_type_eng': 'FAP'},
 {'facility_title': 'Адамовский ФАП',
  'link': 'https://healthcare.by/instinfo.php?orgnum=17185',
  'settlement': 'Адамово',
  'facility_type': 'ФАП',
  'facility_type_eng': 'FAP'},
 {'facility_title': 'Адаховщинский ФАП',
  'link': 'https://healthcare.by/instinfo.php?orgnum=18082',
  'settlement': 'Адаховщина',
  'facility_type': 'ФАП',
  'facility_type_eng': 'FAP'},
 {'facility_title': 'Азделинский ФАП',
  'link': 'https://healthcare.by/instinfo.php?orgnum=15701',
  'settlement': 'Азделино',
  'facility_type': 'ФАП',
  'facility_type_eng': 'FAP'},
 {'facility_title': 'Азинский ФАП',
  'link': 'https://healthcare.by/instinfo.php?orgnum=15299',
  'settlement': 'Азино',
  'facility_type': 'ФАП',
  'facility_type_eng': 'FAP'},
 {'facility_title': 'Александровский ФАП',
  'link': 'https://healthcare.by/instin

In [43]:
# base_url = fap
# # Открываем первую страницу и определяем количество страниц
# response = requests.get(f"{base_url}1")
# doc = BeautifulSoup(response.text, "html.parser")

# page_numbers = []

# for link in doc.select('a[href*="page="]'):
#     match = re.search(r"page=(\d+)", link.get("href", ""))

#     if match:
#         page_numbers.append(int(match.group(1)))

# last_page = max(page_numbers, default=1)

# print("Количество страниц:", last_page)

Количество страниц: 105


In [45]:
rows = []

for category_url in links:
    base_url = category_url.split("&page=")[0]

    response = requests.get(base_url, params={"page": 1})
    doc = BeautifulSoup(response.text, "html.parser")

    page_numbers = []

    for page_link in doc.select('a[href*="page="]'):
        match = re.search(r"page=(\d+)", page_link.get("href", ""))

        if match:
            page_numbers.append(int(match.group(1)))

    last_page = max(page_numbers, default=1)
    print("Количество страниц:", last_page)

    for page in range(1, last_page + 1):
        response = requests.get(
            base_url,
            params={"page": page},
            timeout=30
        )
        response.raise_for_status()

        print(response.url)

        doc = BeautifulSoup(response.text, "html.parser")
        items = doc.select("li")

        for item in items:
            facility_link = item.select_one("a")

            if facility_link is None:
                continue

            name = facility_link.get_text(" ", strip=True)
            facility_type, type_eng = get_facility_type(name)

            row = {
                "facility_title": name,
                "link": f"https://healthcare.by/{facility_link['href']}",
                "settlement": facility_link.next_sibling.strip(" \xa0()"),
                "facility_type": facility_type,
                "facility_type_eng": type_eng,
            }

            rows.append(row)

Количество страниц: 105
https://healthcare.by/search.php?typ=707&sort=1&page=1
https://healthcare.by/search.php?typ=707&sort=1&page=2
https://healthcare.by/search.php?typ=707&sort=1&page=3
https://healthcare.by/search.php?typ=707&sort=1&page=4
https://healthcare.by/search.php?typ=707&sort=1&page=5
https://healthcare.by/search.php?typ=707&sort=1&page=6
https://healthcare.by/search.php?typ=707&sort=1&page=7
https://healthcare.by/search.php?typ=707&sort=1&page=8
https://healthcare.by/search.php?typ=707&sort=1&page=9
https://healthcare.by/search.php?typ=707&sort=1&page=10
https://healthcare.by/search.php?typ=707&sort=1&page=11
https://healthcare.by/search.php?typ=707&sort=1&page=12
https://healthcare.by/search.php?typ=707&sort=1&page=13
https://healthcare.by/search.php?typ=707&sort=1&page=14
https://healthcare.by/search.php?typ=707&sort=1&page=15
https://healthcare.by/search.php?typ=707&sort=1&page=16
https://healthcare.by/search.php?typ=707&sort=1&page=17
https://healthcare.by/search.php?

In [46]:
len(rows)

3501

In [47]:
df = pd.DataFrame(rows)

In [48]:
df.duplicated().sum()

np.int64(4)

In [49]:
# Фильтруем df, чтобы оставить все строки, которые где-либо повторяются (keep=False)
duplicates_df = df[df.duplicated(keep=False)]

# Сортируем их по всем колонкам для визуального объединения пар
duplicates_df = duplicates_df.sort_values(by=duplicates_df.columns.tolist())

# Показываем результат
duplicates_df

,facility_title,link,settlement,facility_type,facility_type_eng
17,Антоновский ФАП,https://healthcare.by/instinfo.php?orgnum=15810,Антоновка,ФАП,FAP
20,Антоновский ФАП,https://healthcare.by/instinfo.php?orgnum=15810,Антоновка,ФАП,FAP
596,Запольский ФАП,https://healthcare.by/instinfo.php?orgnum=15285,Заполье,ФАП,FAP
600,Запольский ФАП,https://healthcare.by/instinfo.php?orgnum=15285,Заполье,ФАП,FAP
1259,Октябрьский ФАП,https://healthcare.by/instinfo.php?orgnum=17342,Октябрь,ФАП,FAP
1260,Октябрьский ФАП,https://healthcare.by/instinfo.php?orgnum=17342,Октябрь,ФАП,FAP
1319,Павловичский ФАП,https://healthcare.by/instinfo.php?orgnum=16201,Павловичи,ФАП,FAP
1320,Павловичский ФАП,https://healthcare.by/instinfo.php?orgnum=16201,Павловичи,ФАП,FAP


In [50]:
df = df.drop_duplicates(subset="link").reset_index(drop=True)

In [51]:
len(df)

3497

In [54]:
rows = df.to_dict("records")

In [55]:
for row in rows:
    # Open the facility page
    response = requests.get(row["link"])
    doc = BeautifulSoup(response.text, "html.parser")

    # Find the first paragraph containing a <b> tag
    info = next(
        (p for p in doc.find_all("p") if p.find("b")),
        None
    )

    # Default values if the data is not available
    address = None
    website = None

    if info:
        # Extract all text parts from the paragraph
        text_parts = list(info.stripped_strings)

        # The first text after the facility name is the address
        if len(text_parts) > 1:
            address = text_parts[1].replace("\xa0", " ").strip()

        # Find an external website link
        website_tag = info.select_one(
            'a[href^="http://"], a[href^="https://"]'
        )

        if website_tag:
            website = website_tag.get("href")

    # Add the extracted values to the existing row
    row["address"] = address
    row["website"] = website

In [56]:
rows

[{'facility_title': 'Агеевский ФАП',
  'link': 'https://healthcare.by/instinfo.php?orgnum=16445',
  'settlement': 'Агеевка',
  'facility_type': 'ФАП',
  'facility_type_eng': 'FAP',
  'address': 'ул. Сиреневая, 9, 213114, д. Агеевка, Могилевский р-н, Могилевская обл.',
  'website': None},
 {'facility_title': 'Адамовский ФАП',
  'link': 'https://healthcare.by/instinfo.php?orgnum=17185',
  'settlement': 'Адамово',
  'facility_type': 'ФАП',
  'facility_type_eng': 'FAP',
  'address': 'д. Адамово, Воложинский р-н, Минская обл.',
  'website': None},
 {'facility_title': 'Адаховщинский ФАП',
  'link': 'https://healthcare.by/instinfo.php?orgnum=18082',
  'settlement': 'Адаховщина',
  'facility_type': 'ФАП',
  'facility_type_eng': 'FAP',
  'address': 'пер. Парковый, 10, 225372, д. Адаховщина, Ляховичский р-н, Брестская обл.',
  'website': None},
 {'facility_title': 'Азделинский ФАП',
  'link': 'https://healthcare.by/instinfo.php?orgnum=15701',
  'settlement': 'Азделино',
  'facility_type': 'ФАП',

In [57]:
df = pd.DataFrame(rows)

In [58]:
df

,facility_title,link,settlement,facility_type,facility_type_eng,address,website
0,Агеевский ФАП,https://healthcare.by/instinfo.php?orgnum=16445,Агеевка,ФАП,FAP,"ул. Сиреневая, 9, 213114, д. Агеевка, Могилевс...",NaN
1,Адамовский ФАП,https://healthcare.by/instinfo.php?orgnum=17185,Адамово,ФАП,FAP,"д. Адамово, Воложинский р-н, Минская обл.",NaN
2,Адаховщинский ФАП,https://healthcare.by/instinfo.php?orgnum=18082,Адаховщина,ФАП,FAP,"пер. Парковый, 10, 225372, д. Адаховщина, Ляхо...",NaN
3,Азделинский ФАП,https://healthcare.by/instinfo.php?orgnum=15701,Азделино,ФАП,FAP,"ул. Новая, 1а, 247025, д. Азделино, Гомельский...",NaN
4,Азинский ФАП,https://healthcare.by/instinfo.php?orgnum=15299,Азино,ФАП,FAP,"211424, д. Азино, Полоцкий р-н, Витебская обл.",NaN
...,...,...,...,...,...,...,...
3492,Филиал № 6 ГУЗ «Гомельская центральная городск...,https://healthcare.by/instinfo.php?orgnum=4673,Гомель,Поликлиника,Polyclinic,"ул. Новополесская, 11, 246003, г. Гомель",NaN
3493,Филиал №1 ГУЗ «ВГПЦ» городская поликлиника №1,https://healthcare.by/instinfo.php?orgnum=26881,Витебск,Поликлиника,Polyclinic,(vgcp.by),NaN
3494,Центральная городская стоматологическая поликл...,https://healthcare.by/instinfo.php?orgnum=5555,Гродно,Поликлиника,Polyclinic,"ул. Суворова, 21, 230001, г. Гродно",http://cgsp.by
3495,Центральная поликлиника Департамента финансов ...,https://healthcare.by/instinfo.php?orgnum=589,Минск,Поликлиника,Polyclinic,"ул. Мясникова, 25а, 220030, г. Минск",http://cpmvd.by


,facility_title,link,settlement,facility_type,facility_type_eng,address,website


In [77]:
# Select rows where both facility type columns are missing
missing_type = (
    df["facility_type"].isna()
    & df["facility_type_eng"].isna()
)

# Find hospital abbreviations:
# ЦКРБ, ЦРБ, ЦГБ, or УБ
hospital_mask = (
    missing_type
    & df["facility_title"].str.contains(
        r"(?<!\w)(?:ЦРКБ|ЦРБ|ЦГБ|УБ)(?!\w)",
        case=False,
        na=False,
        regex=True
    )
)

# Update hospital values
df.loc[
    hospital_mask,
    ["facility_type", "facility_type_eng"]
] = ["Больница", "Hospital"]


# Recalculate the missing-value mask after updating hospitals
missing_type = (
    df["facility_type"].isna()
    & df["facility_type_eng"].isna()
)

# Find outpatient clinic names and abbreviations
ambulatory_mask = (
    missing_type
    & (
        df["facility_title"].str.contains(
            r"(?<!\w)(?:АВОП|ВАП|АОП)(?!\w)"
            r"|БСУ\s+с\s+ВА"
            r"|врача\s+общей\s+практики",
            case=False,
            na=False,
            regex=True
        )
        |
        # Match standalone "ВА" only when written in uppercase
        df["facility_title"].str.contains(
            r"(?<!\w)ВА(?!\w)",
            case=True,
            na=False,
            regex=True
        )
    )
)

# Update outpatient clinic values
df.loc[
    ambulatory_mask,
    ["facility_type", "facility_type_eng"]
] = ["Амбулатория", "Outpatient Clinic"]

In [79]:
# Remove rows where either facility type column is still missing
df = df.dropna(
    subset=["facility_type", "facility_type_eng"]
).reset_index(drop=True)

In [84]:
# Remove rows where address column is still missing
df = df.dropna(
    subset=["address"]
).reset_index(drop=True)

In [85]:
df[df['address'].isna()]

,facility_title,link,settlement,facility_type,facility_type_eng,address,website


## Checking scraped addresses

In [87]:
# Pattern for a domain appearing anywhere in the address
website_pattern = r"(?:https?://|www\.)?(?:[a-z0-9-]+\.)+[a-z]{2,}"

# Find rows where the address contains a website/domain
wrong_address_mask = df["address"].str.contains(
    website_pattern,
    case=False,
    na=False,
    regex=True
)

# Show suspicious rows
df.loc[
    wrong_address_mask,
    ["facility_title", "link", "address", "website"]
]

,facility_title,link,address,website
3471,Филиал №1 ГУЗ «ВГПЦ» городская поликлиника №1,https://healthcare.by/instinfo.php?orgnum=26881,(vgcp.by),NaN


In [101]:
import re

# Common address markers
address_markers = [
    "ул.",
    "пр.",
    "пр-т",
    "пер.",
    "пл.",
    "д.",
    "аг.",
    "агр.",
    "г.п.",
    "г.",
    "обл.",
    "р-н",
    "с/с",
    "п.",
]

# Build a regular expression for address markers
address_pattern = "|".join(
    re.escape(marker) for marker in address_markers
)

# Patterns that often indicate contact details or other non-address text
junk_pattern = (
    r"тел\.?|телефон|факс|"
    r"https?://|www\.|@|"
    r"\bГУЗ\b|\bУЗ\b"
)

# A Belarusian postal code usually contains 6 digits
postal_code_pattern = r"\b\d{6}\b"

# Address looks valid if it contains an address marker or postal code
looks_like_address = (
    df["address"].str.contains(
        address_pattern,
        case=False,
        na=False,
        regex=True
    )
    |
    df["address"].str.contains(
        postal_code_pattern,
        na=False,
        regex=True
    )
)

# Detect phone numbers, websites, email addresses, and institution names
looks_like_junk = df["address"].str.contains(
    junk_pattern,
    case=False,
    na=False,
    regex=True
)

# Find non-empty values that do not contain
# any address marker or postal code
suspicious_address_mask = (
    df["address"].notna()
    & ~looks_like_address
)

df.loc[
    suspicious_address_mask,
    ["facility_title", "link", "address", "website"]
]['address']

638               тел.: (02345) 3-66-88
2253              тел.: (01713) 7-60-18
2625             тел.: (0225)  43-68-52
2799    тел.: (0177) 95-61-04, 95-61-07
Name: address, dtype: str

In [99]:
suspicious_address_mask.sum()

np.int64(4)

In [100]:
import re

# Address markers
address_markers = [
    "ул.", "пр.", "пр-т", "пер.", "пл.", "д.",
    "аг.", "агр.", "г.п.", "г.", "обл.",
    "р-н", "с/с", "п."
]

address_pattern = "|".join(
    re.escape(marker) for marker in address_markers
)

postal_code_pattern = r"\b\d{6}\b"

# Re-scrape only suspicious rows
for index in df.index[suspicious_address_mask]:
    response = requests.get(df.at[index, "link"], timeout=30)
    response.raise_for_status()

    doc = BeautifulSoup(response.text, "html.parser")

    # Find the main facility information paragraph
    info = next(
        (p for p in doc.find_all("p") if p.find("b")),
        None
    )

    if info is None:
        continue

    # Extract all text fragments from the paragraph
    text_parts = [
        text.replace("\xa0", " ").strip(" ()")
        for text in info.stripped_strings
    ]

    # Keep only fragments that look like an address
    address_candidates = [
        text for text in text_parts
        if (
            re.search(postal_code_pattern, text)
            or re.search(address_pattern, text, flags=re.IGNORECASE)
        )
    ]

    # Use the first address-like fragment
    if address_candidates:
        df.at[index, "address"] = address_candidates[0]

    # Extract the website if it is a clickable link
    website_tag = info.select_one(
        'a[href^="http://"], a[href^="https://"]'
    )

    if website_tag:
        df.at[index, "website"] = website_tag.get("href")

In [97]:
df.loc[
    suspicious_address_mask,
    ["facility_title", "address", "website"]
]

,facility_title,address,website
638,Золотушский ФАП,тел.: (02345) 3-66-88,NaN
1008,"Ляховщинский ФАП, ФАП","211875, д. Ляховщина, Поставский р-н, Витебска...",NaN
1405,"Половский ФАП, Учреждение Здравоохранения «Пос...","211875, д. Полово, Поставский р-н, Витебская обл.",NaN
1857,Устянский ФАП,"Приозерная, 17, 213540, д. Устье, Чериковский ...",NaN
2103,"Амбулатория по ул. Славинского, 5, амбулатория","Амбулатория по ул. Славинского, 5",NaN
2253,Дубровская врачебная амбулатория общей практики,тел.: (01713) 7-60-18,NaN
2625,Титовская амбулатория врача общей практики,тел.: (0225) 43-68-52,NaN
2682,"Щорсовская амбулатория врача общей практики, а...","ул. Парковая, 23а, 231428, д. Щорсы, Новогрудс...",NaN
2799,Ганцевичская участковая больница,"тел.: (0177) 95-61-04, 95-61-07",NaN
3094,"Пышнянская больница сестринского ухода, больница","211199, д. Пышно, Лепельский р-н, Витебская обл.",NaN


In [ ]:
повторная обработка

In [103]:
import re
import requests
from bs4 import BeautifulSoup

# Normalize text for comparison
title_text = df["facility_title"].fillna("").str.strip().str.lower()
address_text = df["address"].fillna("").str.strip().str.lower()

# Detect phone numbers or websites stored as an address
phone_or_website = df["address"].str.contains(
    r"^\s*(?:тел\.?|телефон|факс)"
    r"|https?://|www\."
    r"|(?:[a-z0-9-]+\.)+[a-z]{2,}",
    case=False,
    na=False,
    regex=True
)

# Detect cases where the address is identical to the facility title
same_as_title = (
    address_text.ne("")
    & address_text.eq(title_text)
)

# Detect institution descriptions without a postal code
institution_text = (
    ~df["address"].str.contains(r"\b\d{6}\b", na=False, regex=True)
    & df["address"].str.contains(
        r"\b(?:ГУЗ|УЗ|ЛПУ|ФАП|поликлиника|больница|амбулатория)\b"
        r"|учреждени[ея]\s+здравоохранения",
        case=False,
        na=False,
        regex=True
    )
)

# Select rows that need to be re-scraped
bad_address_mask = (
    phone_or_website
    | same_as_title
    | institution_text
)

df.loc[
    bad_address_mask,
    ["facility_title", "address", "link"]
]

,facility_title,address,link
638,Золотушский ФАП,тел.: (02345) 3-66-88,https://healthcare.by/instinfo.php?orgnum=15855
2253,Дубровская врачебная амбулатория общей практики,тел.: (01713) 7-60-18,https://healthcare.by/instinfo.php?orgnum=14134
2625,Титовская амбулатория врача общей практики,тел.: (0225) 43-68-52,https://healthcare.by/instinfo.php?orgnum=24959
2799,Ганцевичская участковая больница,"тел.: (0177) 95-61-04, 95-61-07",https://healthcare.by/instinfo.php?orgnum=6175
3356,Детская поликлиника г. Слоним,Детская поликлиника г. Слоним,https://healthcare.by/instinfo.php?orgnum=5905
3361,Детская стоматологическая поликлиника г. Гродно,Филиал учреждения здравоохранения «Центральная...,https://healthcare.by/instinfo.php?orgnum=5556


In [104]:
for index in df.index[bad_address_mask]:
    response = requests.get(df.at[index, "link"], timeout=30)
    response.raise_for_status()

    doc = BeautifulSoup(response.text, "html.parser")

    # Find the main facility information paragraph
    info = next(
        (p for p in doc.find_all("p") if p.find("b")),
        None
    )

    if info is None:
        df.at[index, "address"] = None
        continue

    # Extract separate text fragments from the paragraph
    text_parts = [
        text.replace("\xa0", " ").strip(" ()")
        for text in info.stripped_strings
    ]

    # Remove empty values, phone numbers and websites
    text_parts = [
        text for text in text_parts
        if text
        and not re.search(
            r"^(?:тел\.?|телефон|факс)"
            r"|https?://|www\."
            r"|(?:[a-z0-9-]+\.)+[a-z]{2,}$",
            text,
            flags=re.IGNORECASE
        )
    ]

    # First choice: a line containing a six-digit postal code
    address_candidates = [
        text for text in text_parts
        if re.search(r"\b\d{6}\b", text)
    ]

    # Fallback: a line containing an address marker and a number
    if not address_candidates:
        address_candidates = [
            text for text in text_parts
            if re.search(
                r"\b(?:ул\.|пр-т|пр\.|пер\.|пл\.|д\.|аг\.|агр\.|"
                r"г\.п\.|г\.|обл\.|р-н|с/с|п\.)",
                text,
                flags=re.IGNORECASE
            )
            and re.search(r"\d", text)
            and text.lower() != title_text.at[index]
        ]

    # Replace the incorrect value or use None if no address was found
    df.at[index, "address"] = (
        address_candidates[0]
        if address_candidates
        else None
    )

    # Extract the facility website when available
    website_tag = info.select_one(
        'a[href^="http://"], a[href^="https://"]'
    )

    if website_tag:
        df.at[index, "website"] = website_tag.get("href")

In [105]:
df.loc[
    df["address"].isna()
    | df["address"].str.contains(
        r"^\s*(?:тел\.?|телефон|факс)",
        case=False,
        na=False,
        regex=True
    ),
    ["facility_title", "address", "link"]
]

,facility_title,address,link
638,Золотушский ФАП,NaN,https://healthcare.by/instinfo.php?orgnum=15855
2253,Дубровская врачебная амбулатория общей практики,NaN,https://healthcare.by/instinfo.php?orgnum=14134
2625,Титовская амбулатория врача общей практики,NaN,https://healthcare.by/instinfo.php?orgnum=24959
2799,Ганцевичская участковая больница,NaN,https://healthcare.by/instinfo.php?orgnum=6175
3361,Детская стоматологическая поликлиника г. Гродно,NaN,https://healthcare.by/instinfo.php?orgnum=5556


In [106]:
df.to_csv("healthcare_facilities_healthcare.by.csv")

# GEOCODING

In [108]:
import os
from dotenv import load_dotenv
from dotenv import dotenv_values
load_dotenv()

GOOGLE_GEO_API_KEY = os.environ["GOOGLE_GEO_API_KEY"]
GOOGLE_PLACES_KEY = os.environ["GOOGLE_PLACES_KEY"]

In [109]:
for address in df["address"]:
    print(address)

ул. Сиреневая, 9, 213114, д. Агеевка, Могилевский р-н, Могилевская обл.
д. Адамово, Воложинский р-н, Минская обл.
пер. Парковый, 10, 225372, д. Адаховщина, Ляховичский р-н, Брестская обл.
ул. Новая, 1а, 247025, д. Азделино, Гомельский р-н, Гомельская обл.
211424, д. Азино, Полоцкий р-н, Витебская обл.
д. 9А, 247507, д. Александровка, Речицкий р-н, Гомельская обл.
ул. Первомайская, 85, 247718, д. Александровка, Калинковичский р-н, Гомельская обл.
ул. Ленина, 18, 225834, д. Алексеевичи, Дрогичинский р-н, Брестская обл.
д. Алексиничи, Сенненский р-н, Витебская обл.
213170, д. Олешковичи, Белыничский р-н, Могилевская обл.
п. Биозавода, Несвижский р-н, Минская обл.
211262, д. Амбросовичи, Шумилинский р-н, Витебская обл.
ул. Молодежная, 5, 213109, д. Амховая 1-я, Могилевский р-н, Могилевская обл.
д. Ананичи, Пуховичский р-н, Минская обл.
ул. Козлова, 1, 223741, д. Ананчицы, Солигорский р-н, Минская обл.
Юньковский с/с, 211875, д. Андроны, Поставский р-н, Витебская обл.
ул. Советская, 44А, 24

# Find lat and long

In [110]:
import geocoder

g = geocoder.google('2950 Broadway, New York, NY', key=GOOGLE_GEO_API_KEY)
g.latlng
from geopy.geocoders import GoogleV3

geolocator = GoogleV3(GOOGLE_GEO_API_KEY)

In [32]:
# df["latitude"] = None
# df["longitude"] = None

# for i, address in enumerate(df["address"]):
#     location = geolocator.geocode(address)
#     if location:
#         df.loc[i, "latitude"] = location.latitude
#         df.loc[i, "longitude"] = location.longitude

In [125]:
# df["latitude"] = None
# df["longitude"] = None

# for i, address in enumerate(df["address"]):
#     if pd.isna(address):
#         address = df.loc[i, "facility_title"]

#     location = geolocator.geocode(address + ", Беларусь")

#     if location:
#         df.loc[i, "latitude"] = location.latitude
#         df.loc[i, "longitude"] = location.longitude

In [130]:
address = "Жарский с/с, ул. Центральная, 18, 211482, д. Жары, Ушачский р-н, Витебская обл."

location = geolocator.geocode(
    address,
    components={"country": "BY"},
    language="ru"
)

print(location.address)
print(location.latitude, location.longitude)

улица Центральная, Віцебская вобласць, Беларусь
55.1860129 29.0410481


In [131]:
df["latitude"] = None
df["longitude"] = None

for i, address in df["address"].items():
    if pd.isna(address):
        address = df.loc[i, "facility_title"]

    location = geolocator.geocode(address,
        components={"country": "BY"},
        language="ru")

    if location:
        df.loc[i, "latitude"] = location.latitude
        df.loc[i, "longitude"] = location.longitude

In [128]:
df.index.equals(pd.RangeIndex(len(df)))

True

In [ ]:
тест яндекса

In [135]:
import os
import requests
from dotenv import load_dotenv

# Reload the current value from .env
load_dotenv(override=True)

api_key = os.getenv("YANDEX_API_KEY", "").strip()

print("Key loaded:", bool(api_key))
print("Key length:", len(api_key))

response = requests.get(
    "https://geocode-maps.yandex.ru/v1/",
    params={
        "apikey": api_key,
        "geocode": "Минск",
        "format": "json"
    },
    timeout=30
)

print("Status:", response.status_code)
print(response.text)

Key loaded: True
Key length: 36
Status: 200
{"response":{"GeoObjectCollection":{"metaDataProperty":{"GeocoderResponseMetaData":{"request":"Минск","results":"10","found":"10"}},"featureMember":[{"GeoObject":{"metaDataProperty":{"GeocoderMetaData":{"precision":"other","text":"Беларусь, Минск","kind":"locality","Address":{"country_code":"BY","formatted":"Беларусь, Минск","Components":[{"kind":"country","name":"Беларусь"},{"kind":"province","name":"Минск"},{"kind":"locality","name":"Минск"}]},"AddressDetails":{"Country":{"AddressLine":"Беларусь, Минск","CountryNameCode":"BY","CountryName":"Беларусь","AdministrativeArea":{"AdministrativeAreaName":"Минск","Locality":{"LocalityName":"Минск"}}}}}},"name":"Минск","description":"Беларусь","boundedBy":{"Envelope":{"lowerCorner":"27.374425 53.793885","upperCorner":"28.080581 53.97162"}},"uri":"ymapsbm1://geo?data=Cgg1MzE3NzAxORIc0JHQtdC70LDRgNGD0YHRjCwg0JzRltC90YHQuiIKDaJ-3EEV8ZtXQg,,","Point":{"pos":"27.561831 53.902284"}}},{"GeoObject":{"metaDat

In [136]:
# Load API key from .env
load_dotenv(override=True)
api_key = os.getenv("YANDEX_API_KEY")

addresses = {
    "Светочский ФАП":
        "ул. Совхозная, 1а, 225210, д. Светоч, Березовский р-н, Брестская обл.",

    "Завечельский ФАП":
        "Сорочинский с/с, ул. Центральная, 21, 211498, д. Завечелье, Ушачский р-н, Витебская обл.",

    "Песковская амбулатория общей практики":
        "пер. Школьный, 5, 231600, д. Пески, Мостовский р-н, Гродненская обл.",

    "Якшанский ФАП":
        "ул. Октябрьская, 20, 225803, д. Якша, Ивановский р-н, Брестская обл.",

    "Жарский ФАП":
        "Жарский с/с, ул. Центральная, 18, 211482, д. Жары, Ушачский р-н, Витебская обл."
}

for title, address in addresses.items():
    response = requests.get(
        "https://geocode-maps.yandex.ru/v1/",
        params={
            "apikey": api_key,
            "geocode": address,
            "format": "json",
            "lang": "ru_RU",
            "bbox": "23.1,51.2~32.8,56.3",
            "rspn": 1,
            "results": 1
        },
        timeout=30
    )

    response.raise_for_status()
    data = response.json()

    results = data["response"]["GeoObjectCollection"]["featureMember"]

    if results:
        geo_object = results[0]["GeoObject"]

        # Yandex returns longitude first, then latitude
        longitude, latitude = geo_object["Point"]["pos"].split()

        metadata = geo_object["metaDataProperty"]["GeocoderMetaData"]

        print(title)
        print("Found address:", metadata.get("text"))
        print("Precision:", metadata.get("precision"))
        print("Latitude:", latitude)
        print("Longitude:", longitude)
        print()
    else:
        print(title)
        print("Not found")
        print()

Светочский ФАП
Found address: Беларусь, Брестская область, Берёзовский сельсовет, деревня Светоч, Совхозная улица, 1А
Precision: exact
Latitude: 52.488707
Longitude: 24.92365

Завечельский ФАП
Found address: Беларусь, Витебская область, Ушачский район, Сорочинский сельсовет, деревня Завечелье, Центральная улица, 21
Precision: exact
Latitude: 55.130024
Longitude: 28.744607

Песковская амбулатория общей практики
Found address: Беларусь, Гродненская область, Мостовский район, агрогородок Лунно, Школьный переулок, 5
Precision: exact
Latitude: 53.455425
Longitude: 24.260756

Якшанский ФАП
Found address: Беларусь, Брестская область, Ивановский район, Бродницкий сельсовет, деревня Якша, Октябрьская улица, 20
Precision: exact
Latitude: 52.164282
Longitude: 25.73614

Жарский ФАП
Found address: Беларусь, Витебская область, Ушачский район, деревня Жары, Центральная улица, 18
Precision: exact
Latitude: 55.078767
Longitude: 28.651973



In [137]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv


# Reload environment variables from the .env file.
# override=True is important in Jupyter because an old key may remain in memory.
load_dotenv(override=True)

api_key = os.getenv("YANDEX_API_KEY")

if not api_key:
    raise ValueError("YANDEX_API_KEY was not found in the .env file")


# Unique links of the facilities that must be geocoded again with Yandex.
# We use links instead of facility names because names may be duplicated,
# while orgnum in each link identifies a specific facility.
test_links = [
    "https://healthcare.by/instinfo.php?orgnum=17198",
    "https://healthcare.by/instinfo.php?orgnum=15458",
    "https://healthcare.by/instinfo.php?orgnum=12328",
    "https://healthcare.by/instinfo.php?orgnum=17822",
    "https://healthcare.by/instinfo.php?orgnum=15457",
]


# Create a Boolean mask:
# True means that the row belongs to one of the selected facilities.
test_mask = df["link"].isin(test_links)


# Check that all requested facilities were found in the DataFrame.
found_links = set(df.loc[test_mask, "link"])
missing_links = set(test_links) - found_links

print(f"Facilities found: {test_mask.sum()} of {len(test_links)}")

if missing_links:
    print("Links not found in the DataFrame:")
    for link in missing_links:
        print(link)


# Optional safety check:
# the same facility link should normally appear only once.
if df.loc[test_mask, "link"].duplicated().any():
    raise ValueError("Duplicate facility links were found in the selected rows")


# df.index[test_mask] returns the real DataFrame index labels
# of the selected facilities.
#
# This is safer than enumerate(), because enumerate() always creates
# numbers 0, 1, 2, 3..., which may not match the actual DataFrame index.
for i in df.index[test_mask]:

    # Read the address from the exact row identified by index label i.
    address = df.at[i, "address"]

    # If the address is missing, use the facility title and settlement
    # as a fallback search query.
    if pd.isna(address) or not str(address).strip():
        address = (
            str(df.at[i, "facility_title"])
            + ", "
            + str(df.at[i, "settlement"])
        )

    # Send the address to the Yandex Geocoder API.
    response = requests.get(
        "https://geocode-maps.yandex.ru/v1/",
        params={
            "apikey": api_key,
            "geocode": address,
            "format": "json",
            "lang": "ru_RU",

            # Approximate bounding box of Belarus.
            "bbox": "23.1,51.2~32.8,56.3",

            # Restrict results to the specified bounding box.
            "rspn": 1,

            # Return only the best matching result.
            "results": 1,
        },
        timeout=30,
    )

    # Stop and show an error if Yandex returns an unsuccessful response.
    response.raise_for_status()

    data = response.json()

    # Extract the list of geocoding results.
    results = data[
        "response"
    ][
        "GeoObjectCollection"
    ][
        "featureMember"
    ]

    if results:
        # Yandex returns coordinates in this order:
        # longitude latitude
        longitude, latitude = (
            results[0]["GeoObject"]["Point"]["pos"].split()
        )

        # Write the coordinates back to the same DataFrame row.
        #
        # df.at[i, ...] uses the real index label i, so the result
        # cannot accidentally be written to another facility.
        df.at[i, "latitude"] = float(latitude)
        df.at[i, "longitude"] = float(longitude)

        print(
            df.at[i, "facility_title"],
            "->",
            latitude,
            longitude
        )

    else:
        print(
            df.at[i, "facility_title"],
            "-> not found"
        )

Facilities found: 5 of 5
Жарский ФАП -> 55.078767 28.651973
Завечельский ФАП -> 55.130024 28.744607
Светочский ФАП -> 52.488707 24.92365
Якшанский ФАП -> 52.164282 25.73614
Песковская амбулатория общей рактики -> 53.455425 24.260756


взять еще 500 рандомных фапов, для которых мы нашли координаты в гугл и проверить их в яндексе, сравнить с гугл, и заодно переписать их в датафрейме - боюсь, даже если гугл нашел в рб, то это может быть не всегда верная локация

In [138]:
import os
import time
import requests
import pandas as pd
from math import radians, sin, cos, sqrt, atan2
from dotenv import load_dotenv


# Reload the Yandex API key from .env
load_dotenv(override=True)
api_key = os.getenv("YANDEX_API_KEY")

if not api_key:
    raise ValueError("YANDEX_API_KEY was not found in .env")


# Calculate the distance between Google and Yandex coordinates in kilometers
def calculate_distance(lat1, lon1, lat2, lon2):
    earth_radius = 6371

    lat1, lon1, lat2, lon2 = map(
        radians,
        [lat1, lon1, lat2, lon2]
    )

    latitude_difference = lat2 - lat1
    longitude_difference = lon2 - lon1

    a = (
        sin(latitude_difference / 2) ** 2
        + cos(lat1)
        * cos(lat2)
        * sin(longitude_difference / 2) ** 2
    )

    return 2 * earth_radius * atan2(sqrt(a), sqrt(1 - a))


# Select FAP rows for which Google found both coordinates
fap_candidates = df[
    (df["facility_type"] == "ФАП")
    & df["latitude"].notna()
    & df["longitude"].notna()
]


# Select up to 500 random FAPs.
# random_state=42 makes the sample reproducible:
# running this cell again selects the same facilities.
sample_size = min(500, len(fap_candidates))

sample_indices = fap_candidates.sample(
    n=sample_size,
    random_state=42
).index


# Store comparison results here.
# This list does not add extra columns to the main DataFrame.
comparison_rows = []


# Use the real DataFrame index for every selected facility.
# This guarantees that Yandex coordinates are written back
# to the same row from which the address was taken.
for i in sample_indices:

    address = df.at[i, "address"]

    # Use facility title and settlement if the address is missing
    if pd.isna(address) or not str(address).strip():
        address = (
            str(df.at[i, "facility_title"])
            + ", "
            + str(df.at[i, "settlement"])
        )

    # Save the existing Google coordinates before replacing them
    google_latitude = float(df.at[i, "latitude"])
    google_longitude = float(df.at[i, "longitude"])

    response = requests.get(
        "https://geocode-maps.yandex.ru/v1/",
        params={
            "apikey": api_key,
            "geocode": address,
            "format": "json",
            "lang": "ru_RU",

            # Approximate boundaries of Belarus
            "bbox": "23.1,51.2~32.8,56.3",

            # Restrict the search to the bounding box
            "rspn": 1,

            # Return only the best result
            "results": 1,
        },
        timeout=30,
    )

    response.raise_for_status()
    data = response.json()

    results = data[
        "response"
    ][
        "GeoObjectCollection"
    ][
        "featureMember"
    ]

    if results:
        geo_object = results[0]["GeoObject"]

        # Yandex returns coordinates as:
        # longitude latitude
        yandex_longitude, yandex_latitude = map(
            float,
            geo_object["Point"]["pos"].split()
        )

        metadata = geo_object[
            "metaDataProperty"
        ][
            "GeocoderMetaData"
        ]

        # Calculate the distance between the old Google point
        # and the new Yandex point
        distance_km = calculate_distance(
            google_latitude,
            google_longitude,
            yandex_latitude,
            yandex_longitude,
        )

        # Save comparison information separately
        comparison_rows.append({
            "df_index": i,
            "facility_title": df.at[i, "facility_title"],
            "address": address,
            "google_latitude": google_latitude,
            "google_longitude": google_longitude,
            "yandex_latitude": yandex_latitude,
            "yandex_longitude": yandex_longitude,
            "distance_km": distance_km,
            "yandex_address": metadata.get("text"),
            "yandex_precision": metadata.get("precision"),
        })

        # Replace Google coordinates in the original DataFrame
        # using the exact original DataFrame index
        df.at[i, "latitude"] = yandex_latitude
        df.at[i, "longitude"] = yandex_longitude

    else:
        comparison_rows.append({
            "df_index": i,
            "facility_title": df.at[i, "facility_title"],
            "address": address,
            "google_latitude": google_latitude,
            "google_longitude": google_longitude,
            "yandex_latitude": None,
            "yandex_longitude": None,
            "distance_km": None,
            "yandex_address": None,
            "yandex_precision": None,
        })

    # Small pause between API requests
    time.sleep(0.1)


# Create a temporary DataFrame for comparison
comparison_df = pd.DataFrame(comparison_rows)

print("Selected FAPs:", sample_size)
print("Found by Yandex:", comparison_df["yandex_latitude"].notna().sum())

Selected FAPs: 500
Found by Yandex: 500


In [139]:
comparison_df.sort_values(
    "distance_km",
    ascending=False
).head(30)

,df_index,facility_title,address,google_latitude,google_longitude,yandex_latitude,yandex_longitude,distance_km,yandex_address,yandex_precision
433,2495,Осиновский ФАП,"ул. Почтовая, 1, 213205, д. Осиновка, Чаусский...",53.943016,27.706895,53.849142,31.217525,230.235885,"Беларусь, Могилёвская область, Чаусский район,...",exact
0,1644,Слободской ФАП,"ул. Школьная, 2, 211328, д. Слобода, Витебский...",55.641159,27.034763,55.398426,30.607291,226.478133,"Беларусь, Витебский район, Задубровский сельсо...",exact
64,1746,Сторожовецкий ФАП,"ул. Советская, 3, 247990, д. Сторожовцы, Житко...",52.327501,30.463163,52.032943,27.783823,185.585556,"Беларусь, Гомельская область, Житковичский рай...",exact
80,122,Боровиковский ФАП,"ул. Ленина, 18, 225656, д. Боровики, Лунинецки...",52.214042,24.358489,52.517256,26.940080,178.487272,"Беларусь, Брестская область, Лунинецкий район,...",exact
466,1076,Милевичский ФАП,"ул. Первомайская, 12, 247976, д. Милевичи, Жит...",52.895237,30.053784,52.466048,27.601887,172.030739,"Беларусь, Гомельская область, Житковичский рай...",exact
140,1225,Новский ФАП,"ул. Гагарина, 30, 225211, д. Новое, Березовски...",52.072421,27.339084,52.483239,25.061146,161.558627,"Беларусь, Брестская область, Берёзовский район...",exact
78,434,Добринский ФАП,"ул. Центральная, 25, 211054, д. Добринь, Дубро...",54.622626,28.706808,54.646698,31.015979,148.631720,"Беларусь, Витебская область, Дубровенский райо...",near
162,1311,Отрадненский ФАП,"ул. Молодежная, 16, 223837, д. Отрадное, Любан...",53.858493,27.689160,52.715426,27.782197,127.253622,"Беларусь, Минская область, Любанский район, Ре...",exact
56,1353,Пересудовичский ФАП,"п/о Высокое, ул. Кирова, 5, 225217, д. Высокое...",52.366211,23.376775,52.401243,25.111209,117.777749,"Беларусь, Брестская область, Берёзовский район...",exact
139,242,Войтовский ФАП,"д. Войтово, Витебский р-н, Витебская обл.",55.295983,28.758363,55.217686,30.518052,111.847594,"Беларусь, Витебский район, Вымнянский сельсове...",other


In [ ]:
проверить новые 500 фапов в яндексе

In [140]:
# Get indexes of facilities already checked with Yandex
checked_indices = set(comparison_df["df_index"])


# Select FAPs that:
# 1. have coordinates;
# 2. have not yet been checked with Yandex.
new_candidates = df[
    (df["facility_type"] == "ФАП")
    & df["latitude"].notna()
    & df["longitude"].notna()
    & ~df.index.isin(checked_indices)
]


# Select up to 500 new random facilities
sample_size = min(500, len(new_candidates))

new_sample_indices = new_candidates.sample(
    n=sample_size,
    random_state=43
).index


# Store comparison results for the new batch
new_comparison_rows = []


for i in new_sample_indices:
    address = df.at[i, "address"]

    # Use facility title and settlement if address is missing
    if pd.isna(address) or not str(address).strip():
        address = (
            str(df.at[i, "facility_title"])
            + ", "
            + str(df.at[i, "settlement"])
        )

    # These are still the original Google coordinates,
    # because this facility has not been checked before
    google_latitude = float(df.at[i, "latitude"])
    google_longitude = float(df.at[i, "longitude"])

    response = requests.get(
        "https://geocode-maps.yandex.ru/v1/",
        params={
            "apikey": api_key,
            "geocode": address,
            "format": "json",
            "lang": "ru_RU",
            "bbox": "23.1,51.2~32.8,56.3",
            "rspn": 1,
            "results": 1,
        },
        timeout=30,
    )

    response.raise_for_status()
    data = response.json()

    results = data[
        "response"
    ][
        "GeoObjectCollection"
    ][
        "featureMember"
    ]

    if results:
        geo_object = results[0]["GeoObject"]

        # Yandex returns longitude first, then latitude
        yandex_longitude, yandex_latitude = map(
            float,
            geo_object["Point"]["pos"].split()
        )

        metadata = geo_object[
            "metaDataProperty"
        ][
            "GeocoderMetaData"
        ]

        distance_km = calculate_distance(
            google_latitude,
            google_longitude,
            yandex_latitude,
            yandex_longitude,
        )

        new_comparison_rows.append({
            "df_index": i,
            "facility_title": df.at[i, "facility_title"],
            "address": address,
            "google_latitude": google_latitude,
            "google_longitude": google_longitude,
            "yandex_latitude": yandex_latitude,
            "yandex_longitude": yandex_longitude,
            "distance_km": distance_km,
            "yandex_address": metadata.get("text"),
            "yandex_precision": metadata.get("precision"),
        })

        # Replace Google coordinates with Yandex coordinates
        # in the same original DataFrame row
        df.at[i, "latitude"] = yandex_latitude
        df.at[i, "longitude"] = yandex_longitude

    else:
        new_comparison_rows.append({
            "df_index": i,
            "facility_title": df.at[i, "facility_title"],
            "address": address,
            "google_latitude": google_latitude,
            "google_longitude": google_longitude,
            "yandex_latitude": None,
            "yandex_longitude": None,
            "distance_km": None,
            "yandex_address": None,
            "yandex_precision": None,
        })

    time.sleep(0.1)


# Create comparison data for this new batch
new_comparison_df = pd.DataFrame(new_comparison_rows)


# Add the new batch to the previous comparison results
comparison_df = pd.concat(
    [comparison_df, new_comparison_df],
    ignore_index=True
)

print("New facilities selected:", sample_size)
print(
    "Found by Yandex:",
    new_comparison_df["yandex_latitude"].notna().sum()
)
print("Total facilities checked:", len(comparison_df))

New facilities selected: 500
Found by Yandex: 500
Total facilities checked: 1000


еще 700 фапов

In [141]:
# Get indexes of all facilities already checked with Yandex
checked_indices = set(comparison_df["df_index"])


# Select FAPs that:
# 1. have coordinates;
# 2. have not yet been checked with Yandex.
new_candidates = df[
    (df["facility_type"] == "ФАП")
    & df["latitude"].notna()
    & df["longitude"].notna()
    & ~df.index.isin(checked_indices)
]


# Select up to 700 new random FAPs
sample_size = min(700, len(new_candidates))

new_sample_indices = new_candidates.sample(
    n=sample_size,
    random_state=44
).index


# Store comparison results for this batch
new_comparison_rows = []


for i in new_sample_indices:
    address = df.at[i, "address"]

    # Use facility title and settlement if the address is missing
    if pd.isna(address) or not str(address).strip():
        address = (
            str(df.at[i, "facility_title"])
            + ", "
            + str(df.at[i, "settlement"])
        )

    # Save the current coordinates before replacing them
    google_latitude = float(df.at[i, "latitude"])
    google_longitude = float(df.at[i, "longitude"])

    response = requests.get(
        "https://geocode-maps.yandex.ru/v1/",
        params={
            "apikey": api_key,
            "geocode": address,
            "format": "json",
            "lang": "ru_RU",
            "bbox": "23.1,51.2~32.8,56.3",
            "rspn": 1,
            "results": 1,
        },
        timeout=30,
    )

    response.raise_for_status()
    data = response.json()

    results = data[
        "response"
    ][
        "GeoObjectCollection"
    ][
        "featureMember"
    ]

    if results:
        geo_object = results[0]["GeoObject"]

        # Yandex returns longitude first, then latitude
        yandex_longitude, yandex_latitude = map(
            float,
            geo_object["Point"]["pos"].split()
        )

        metadata = geo_object[
            "metaDataProperty"
        ][
            "GeocoderMetaData"
        ]

        distance_km = calculate_distance(
            google_latitude,
            google_longitude,
            yandex_latitude,
            yandex_longitude,
        )

        new_comparison_rows.append({
            "df_index": i,
            "facility_title": df.at[i, "facility_title"],
            "address": address,
            "google_latitude": google_latitude,
            "google_longitude": google_longitude,
            "yandex_latitude": yandex_latitude,
            "yandex_longitude": yandex_longitude,
            "distance_km": distance_km,
            "yandex_address": metadata.get("text"),
            "yandex_precision": metadata.get("precision"),
        })

        # Replace the current coordinates with Yandex coordinates
        # in the same original DataFrame row
        df.at[i, "latitude"] = yandex_latitude
        df.at[i, "longitude"] = yandex_longitude

    else:
        new_comparison_rows.append({
            "df_index": i,
            "facility_title": df.at[i, "facility_title"],
            "address": address,
            "google_latitude": google_latitude,
            "google_longitude": google_longitude,
            "yandex_latitude": None,
            "yandex_longitude": None,
            "distance_km": None,
            "yandex_address": None,
            "yandex_precision": None,
        })

    time.sleep(0.1)


# Create comparison data for the new batch
new_comparison_df = pd.DataFrame(new_comparison_rows)


# Add the new batch to all previous comparison results
comparison_df = pd.concat(
    [comparison_df, new_comparison_df],
    ignore_index=True
)


print("New facilities selected:", sample_size)
print(
    "Found by Yandex:",
    new_comparison_df["yandex_latitude"].notna().sum()
)
print("Total facilities checked:", len(comparison_df))

New facilities selected: 700
Found by Yandex: 700
Total facilities checked: 1700


In [142]:
len(df)

3474

In [144]:
df.groupby('facility_type').size()

facility_type
Амбулатория     671
Больница        422
Поликлиника     265
ФАП            2116
dtype: int64

In [ ]:
все оставшиеся фапы

In [145]:
# Get indexes of all facilities already checked with Yandex
checked_indices = set(comparison_df["df_index"])


# Select every remaining FAP that:
# 1. has coordinates;
# 2. has not yet been checked with Yandex.
remaining_faps = df[
    (df["facility_type"] == "ФАП")
    & df["latitude"].notna()
    & df["longitude"].notna()
    & ~df.index.isin(checked_indices)
]

remaining_indices = remaining_faps.index

print("Remaining FAPs:", len(remaining_indices))


# Store comparison results for this batch
new_comparison_rows = []


for i in remaining_indices:
    address = df.at[i, "address"]

    # Use facility title and settlement if the address is missing
    if pd.isna(address) or not str(address).strip():
        address = (
            str(df.at[i, "facility_title"])
            + ", "
            + str(df.at[i, "settlement"])
        )

    # Save the current Google coordinates before replacing them
    google_latitude = float(df.at[i, "latitude"])
    google_longitude = float(df.at[i, "longitude"])

    response = requests.get(
        "https://geocode-maps.yandex.ru/v1/",
        params={
            "apikey": api_key,
            "geocode": address,
            "format": "json",
            "lang": "ru_RU",
            "bbox": "23.1,51.2~32.8,56.3",
            "rspn": 1,
            "results": 1,
        },
        timeout=30,
    )

    response.raise_for_status()
    data = response.json()

    results = data[
        "response"
    ][
        "GeoObjectCollection"
    ][
        "featureMember"
    ]

    if results:
        geo_object = results[0]["GeoObject"]

        # Yandex returns longitude first, then latitude
        yandex_longitude, yandex_latitude = map(
            float,
            geo_object["Point"]["pos"].split()
        )

        metadata = geo_object[
            "metaDataProperty"
        ][
            "GeocoderMetaData"
        ]

        # Calculate the distance between Google and Yandex coordinates
        distance_km = calculate_distance(
            google_latitude,
            google_longitude,
            yandex_latitude,
            yandex_longitude,
        )

        new_comparison_rows.append({
            "df_index": i,
            "facility_title": df.at[i, "facility_title"],
            "facility_type": df.at[i, "facility_type"],
            "address": address,
            "google_latitude": google_latitude,
            "google_longitude": google_longitude,
            "yandex_latitude": yandex_latitude,
            "yandex_longitude": yandex_longitude,
            "distance_km": distance_km,
            "yandex_address": metadata.get("text"),
            "yandex_precision": metadata.get("precision"),
        })

        # Replace Google coordinates in the original DataFrame
        # using the exact index of this facility
        df.at[i, "latitude"] = yandex_latitude
        df.at[i, "longitude"] = yandex_longitude

    else:
        # Keep Google coordinates if Yandex found nothing
        new_comparison_rows.append({
            "df_index": i,
            "facility_title": df.at[i, "facility_title"],
            "facility_type": df.at[i, "facility_type"],
            "address": address,
            "google_latitude": google_latitude,
            "google_longitude": google_longitude,
            "yandex_latitude": None,
            "yandex_longitude": None,
            "distance_km": None,
            "yandex_address": None,
            "yandex_precision": None,
        })

    time.sleep(0.1)


# Add this batch to the previous comparison results
new_comparison_df = pd.DataFrame(new_comparison_rows)

comparison_df = pd.concat(
    [comparison_df, new_comparison_df],
    ignore_index=True
)


print("Processed:", len(new_comparison_df))
print(
    "Found by Yandex:",
    new_comparison_df["yandex_latitude"].notna().sum()
)
print(
    "FAPs still unchecked:",
    (
        (df["facility_type"] == "ФАП")
        & ~df.index.isin(comparison_df["df_index"])
    ).sum()
)

Remaining FAPs: 416
Processed: 416
Found by Yandex: 416
FAPs still unchecked: 0


все амбулатории

In [146]:
# Select all outpatient clinics
ambulatory_indices = df.index[
    df["facility_type"] == "Амбулатория"
]

print("Outpatient clinics to process:", len(ambulatory_indices))


# Store comparison results for this batch
new_comparison_rows = []


for i in ambulatory_indices:
    address = df.at[i, "address"]

    # Use facility title and settlement if the address is missing
    if pd.isna(address) or not str(address).strip():
        address = (
            str(df.at[i, "facility_title"])
            + ", "
            + str(df.at[i, "settlement"])
        )

    # Save the current Google coordinates before replacing them
    google_latitude = df.at[i, "latitude"]
    google_longitude = df.at[i, "longitude"]

    response = requests.get(
        "https://geocode-maps.yandex.ru/v1/",
        params={
            "apikey": api_key,
            "geocode": address,
            "format": "json",
            "lang": "ru_RU",
            "bbox": "23.1,51.2~32.8,56.3",
            "rspn": 1,
            "results": 1,
        },
        timeout=30,
    )

    response.raise_for_status()
    data = response.json()

    results = data["response"]["GeoObjectCollection"]["featureMember"]

    if results:
        geo_object = results[0]["GeoObject"]

        # Yandex returns longitude first, then latitude
        yandex_longitude, yandex_latitude = map(
            float,
            geo_object["Point"]["pos"].split()
        )

        metadata = geo_object["metaDataProperty"]["GeocoderMetaData"]

        # Calculate distance only if Google coordinates exist
        if pd.notna(google_latitude) and pd.notna(google_longitude):
            distance_km = calculate_distance(
                float(google_latitude),
                float(google_longitude),
                yandex_latitude,
                yandex_longitude,
            )
        else:
            distance_km = None

        new_comparison_rows.append({
            "df_index": i,
            "facility_title": df.at[i, "facility_title"],
            "facility_type": df.at[i, "facility_type"],
            "address": address,
            "google_latitude": google_latitude,
            "google_longitude": google_longitude,
            "yandex_latitude": yandex_latitude,
            "yandex_longitude": yandex_longitude,
            "distance_km": distance_km,
            "yandex_address": metadata.get("text"),
            "yandex_precision": metadata.get("precision"),
        })

        # Replace coordinates in the same original DataFrame row
        df.at[i, "latitude"] = yandex_latitude
        df.at[i, "longitude"] = yandex_longitude

    else:
        # Keep the existing coordinates if Yandex found nothing
        new_comparison_rows.append({
            "df_index": i,
            "facility_title": df.at[i, "facility_title"],
            "facility_type": df.at[i, "facility_type"],
            "address": address,
            "google_latitude": google_latitude,
            "google_longitude": google_longitude,
            "yandex_latitude": None,
            "yandex_longitude": None,
            "distance_km": None,
            "yandex_address": None,
            "yandex_precision": None,
        })

    time.sleep(0.1)


# Add this batch to the comparison table
new_comparison_df = pd.DataFrame(new_comparison_rows)

comparison_df = pd.concat(
    [comparison_df, new_comparison_df],
    ignore_index=True
)


print("Processed:", len(new_comparison_df))
print(
    "Found by Yandex:",
    new_comparison_df["yandex_latitude"].notna().sum()
)

Outpatient clinics to process: 671


HTTPError: 403 Client Error: Forbidden for url: https://geocode-maps.yandex.ru/v1/?apikey=7e986c4a-dae7-4a27-afb0-ca02548d2a92&geocode=%D0%BF%D0%B5%D1%80.+%D0%97%D0%B5%D0%BB%D0%B5%D0%BD%D1%8B%D0%B9%2C+4%D0%B0%2C+225811%2C+%D0%B4.+%D0%A0%D1%83%D0%B4%D1%81%D0%BA%2C+%D0%98%D0%B2%D0%B0%D0%BD%D0%BE%D0%B2%D1%81%D0%BA%D0%B8%D0%B9+%D1%80-%D0%BD%2C+%D0%91%D1%80%D0%B5%D1%81%D1%82%D1%81%D0%BA%D0%B0%D1%8F+%D0%BE%D0%B1%D0%BB.&format=json&lang=ru_RU&bbox=23.1%2C51.2~32.8%2C56.3&rspn=1&results=1

In [147]:
# Convert successfully processed rows from this interrupted batch
partial_comparison_df = pd.DataFrame(new_comparison_rows)

# Add only rows that are not already present in comparison_df
partial_comparison_df = partial_comparison_df[
    ~partial_comparison_df["df_index"].isin(comparison_df["df_index"])
]

comparison_df = pd.concat(
    [comparison_df, partial_comparison_df],
    ignore_index=True
)

print("Processed before the error:", len(partial_comparison_df))

Processed before the error: 461


In [148]:
# Find outpatient clinics that have not yet been processed by Yandex
remaining_ambulatory = df[
    (df["facility_type"] == "Амбулатория")
    & ~df.index.isin(comparison_df["df_index"])
]

print("Outpatient clinics remaining:", len(remaining_ambulatory))

Outpatient clinics remaining: 210


In [133]:
#df.to_csv("healthcare_facilities_healthcare.by_with_coordinates3.csv")

In [150]:
comparison_df.to_csv("gathered_yandex_coordinates.csv")

In [149]:
df.to_csv("healthcare_with_coordinates_with_yandex.csv")

In [151]:
df

,facility_title,link,settlement,facility_type,facility_type_eng,address,website,latitude,longitude
0,Агеевский ФАП,https://healthcare.by/instinfo.php?orgnum=16445,Агеевка,ФАП,FAP,"ул. Сиреневая, 9, 213114, д. Агеевка, Могилевс...",NaN,53.959718,30.48202
1,Адамовский ФАП,https://healthcare.by/instinfo.php?orgnum=17185,Адамово,ФАП,FAP,"д. Адамово, Воложинский р-н, Минская обл.",NaN,54.138781,26.308645
2,Адаховщинский ФАП,https://healthcare.by/instinfo.php?orgnum=18082,Адаховщина,ФАП,FAP,"пер. Парковый, 10, 225372, д. Адаховщина, Ляхо...",NaN,53.153936,26.180716
3,Азделинский ФАП,https://healthcare.by/instinfo.php?orgnum=15701,Азделино,ФАП,FAP,"ул. Новая, 1а, 247025, д. Азделино, Гомельский...",NaN,52.540376,30.744805
4,Азинский ФАП,https://healthcare.by/instinfo.php?orgnum=15299,Азино,ФАП,FAP,"211424, д. Азино, Полоцкий р-н, Витебская обл.",NaN,55.623392,28.643457
...,...,...,...,...,...,...,...,...,...
3469,Филиал № 6 ГУЗ «Гомельская центральная городск...,https://healthcare.by/instinfo.php?orgnum=4744,Гомель,Поликлиника,Polyclinic,"пр. Победы, 6, 246022, г. Гомель",http://https//gcgdp.by/ru/o-filiale-6/administ...,52.434359,31.005047
3470,Филиал № 6 ГУЗ «Гомельская центральная городск...,https://healthcare.by/instinfo.php?orgnum=4673,Гомель,Поликлиника,Polyclinic,"ул. Новополесская, 11, 246003, г. Гомель",NaN,52.440964,30.999183
3471,Филиал №1 ГУЗ «ВГПЦ» городская поликлиника №1,https://healthcare.by/instinfo.php?orgnum=26881,Витебск,Поликлиника,Polyclinic,"ул. Терешковой, 30, 210009, г. Витебск",NaN,55.193559,30.248739
3472,Центральная городская стоматологическая поликл...,https://healthcare.by/instinfo.php?orgnum=5555,Гродно,Поликлиника,Polyclinic,"ул. Суворова, 21, 230001, г. Гродно",http://cgsp.by,53.66036,23.812129


In [152]:
comparison_df

,df_index,facility_title,address,google_latitude,google_longitude,yandex_latitude,yandex_longitude,distance_km,yandex_address,yandex_precision,facility_type
0,1644,Слободской ФАП,"ул. Школьная, 2, 211328, д. Слобода, Витебский...",55.641159,27.034763,55.398426,30.607291,226.478133,"Беларусь, Витебский район, Задубровский сельсо...",exact,NaN
1,1495,Рассветовский ФАП,"ул. Центральная, 39б, 223722, д. Рассвет, Соли...",52.815878,27.536414,52.830161,27.251562,19.205774,"Беларусь, Минская область, Солигорский район, ...",exact,NaN
2,262,Воронский ФАП,"ул. Полоцкая, 3, 231221, д. Ворона, Островецки...",54.757295,25.956963,54.637814,25.985135,13.408403,"Беларусь, Гродненская область, Островецкий рай...",exact,NaN
3,437,Добромысленский ФАП,"д. Добромысель, Ивацевичский р-н, Брестская обл.",52.824549,25.708395,52.824437,25.713889,0.369361,"Беларусь, Брестская область, Ивацевичский райо...",other,NaN
4,1530,Рудаяворский ФАП,"231486, д. Руда Яворская, Дятловский р-н, Грод...",53.404221,25.111776,53.402911,25.111461,0.147100,"Беларусь, Гродненская область, Дятловский райо...",other,NaN
...,...,...,...,...,...,...,...,...,...,...,...
2572,2565,Роднянская амбулатория врача общей практики,"ул. Больничная, 213621, д. Родня, Климовичский...",53.514216,32.137208,53.610957,32.089035,11.217737,"Беларусь, Могилёвская область, Климовичский район",other,Амбулатория
2573,2566,Рожанковская амбулатория врача общей практики,"ул. Щучинская, 22а, 231500, д. Рожанка, Щучинс...",53.538879,24.740801,53.525627,24.705153,2.778886,"Беларусь, Гродненская область, Щучинский район...",number,Амбулатория
2574,2568,Рудня-Маримоновская АОП,"ул. Центральная, 5, 247033, д. Рудня-Маримонов...",52.273161,30.730107,52.162935,30.716166,12.293312,"Беларусь, Гомельский район, деревня Рудня Мари...",other,Амбулатория
2575,2569,Руднянская АВОП,"ул. Молодежная, 3, 247755, д. Малая Рудня, Моз...",52.054806,29.226198,51.883584,29.198975,19.130112,"Беларусь, Гомельская область, Мозырский район,...",exact,Амбулатория


повторно перенести все сохранённые координаты из comparison_df в основной df

In [153]:
# Keep the latest successful Yandex result for each original df row
updates = (
    comparison_df
    .dropna(subset=["yandex_latitude", "yandex_longitude"])
    .drop_duplicates(subset="df_index", keep="last")
    .copy()
)

# Keep only indexes that still exist in df
updates = updates[
    updates["df_index"].isin(df.index)
]


# Verify that df indexes still belong to the same facilities
index_check = updates[["df_index", "facility_title"]].copy()

index_check["current_df_title"] = index_check["df_index"].map(
    df["facility_title"]
)

mismatches = index_check[
    index_check["facility_title"] != index_check["current_df_title"]
]

print("Yandex updates found:", len(updates))
print("Index mismatches:", len(mismatches))

Yandex updates found: 2577
Index mismatches: 0


In [154]:
# Use df_index as the index of the update table
updates = updates.set_index("df_index")

# Write all successful Yandex coordinates back into df
df.loc[updates.index, "latitude"] = pd.to_numeric(
    updates["yandex_latitude"],
    errors="coerce"
)

df.loc[updates.index, "longitude"] = pd.to_numeric(
    updates["yandex_longitude"],
    errors="coerce"
)

print("Coordinates updated:", len(updates))

Coordinates updated: 2577


In [155]:
df.loc[
    df["facility_title"].str.contains(
        "Завечельский",
        case=False,
        na=False
    ),
    [
        "facility_title",
        "address",
        "latitude",
        "longitude"
    ]
]

,facility_title,address,latitude,longitude
555,Завечельский ФАП,"Сорочинский с/с, ул. Центральная, 21, 211498, ...",55.130024,28.744607


Points outside Belarus

In [156]:
latitude = pd.to_numeric(df["latitude"], errors="coerce")
longitude = pd.to_numeric(df["longitude"], errors="coerce")

outside_belarus = df[
    latitude.notna()
    & longitude.notna()
    & (
        (latitude < 51.2)
        | (latitude > 56.3)
        | (longitude < 23.1)
        | (longitude > 32.8)
    )
]

print("Points outside Belarus:", len(outside_belarus))

outside_belarus[
    [
        "facility_title",
        "facility_type",
        "address",
        "latitude",
        "longitude"
    ]
]

Points outside Belarus: 0


,facility_title,facility_type,address,latitude,longitude


In [157]:
df.to_csv("healthcare_with_coordinates_with_yandex.csv")

In [159]:
df.columns

Index(['facility_title', 'link', 'settlement', 'facility_type',
       'facility_type_eng', 'address', 'website', 'latitude', 'longitude'],
      dtype='str')

# Добавляем области и районы

In [160]:
import pandas as pd
import geopandas as gpd


# Keep all original columns
df_geo = df.copy()


# Ensure coordinates are numeric
df_geo["latitude"] = pd.to_numeric(
    df_geo["latitude"],
    errors="coerce",
)

df_geo["longitude"] = pd.to_numeric(
    df_geo["longitude"],
    errors="coerce",
)


# Create point geometry
# X = longitude, Y = latitude
df_gdf = gpd.GeoDataFrame(
    df_geo,
    geometry=gpd.points_from_xy(
        df_geo["longitude"],
        df_geo["latitude"],
    ),
    crs="EPSG:4326",
)


# Read administrative districts
adm2 = gpd.read_file(
    "gadm41_BLR.gpkg",
    layer="ADM_ADM_2",
)

adm2 = adm2[
    ["NAME_1", "NAME_2", "geometry"]
].copy()


# Match coordinate systems
adm2 = adm2.to_crs(df_gdf.crs)


# Add region and district to every facility
df_gdf = gpd.sjoin(
    df_gdf,
    adm2,
    how="left",
    predicate="within",
)


# Remove technical join column
df_gdf = df_gdf.drop(
    columns="index_right",
    errors="ignore",
)


# Rename administrative columns
df_gdf = df_gdf.rename(
    columns={
        "NAME_1": "region",
        "NAME_2": "district",
    }
)


# Update the original DataFrame
df["region"] = df_gdf["region"]
df["district"] = df_gdf["district"]


# Save GeoPackage with all columns and geometry
df_gdf.to_file(
    "healthcare_regions.gpkg",
    layer="healthcare",
    driver="GPKG",
    index=False,
)


# Save CSV with all original columns plus region and district
df_gdf.drop(columns="geometry").to_csv(
    "healthcare_regions.csv",
    index=False,
    encoding="utf-8-sig",
)

In [161]:
df

,facility_title,link,settlement,facility_type,facility_type_eng,address,website,latitude,longitude,region,district
0,Агеевский ФАП,https://healthcare.by/instinfo.php?orgnum=16445,Агеевка,ФАП,FAP,"ул. Сиреневая, 9, 213114, д. Агеевка, Могилевс...",NaN,53.959718,30.48202,Mogilev,Mogilev
1,Адамовский ФАП,https://healthcare.by/instinfo.php?orgnum=17185,Адамово,ФАП,FAP,"д. Адамово, Воложинский р-н, Минская обл.",NaN,54.138781,26.308645,Minsk,Valozhyn
2,Адаховщинский ФАП,https://healthcare.by/instinfo.php?orgnum=18082,Адаховщина,ФАП,FAP,"пер. Парковый, 10, 225372, д. Адаховщина, Ляхо...",NaN,53.153936,26.180716,Brest,Baranavichy
3,Азделинский ФАП,https://healthcare.by/instinfo.php?orgnum=15701,Азделино,ФАП,FAP,"ул. Новая, 1а, 247025, д. Азделино, Гомельский...",NaN,52.540376,30.744805,Gomel,Gomyel
4,Азинский ФАП,https://healthcare.by/instinfo.php?orgnum=15299,Азино,ФАП,FAP,"211424, д. Азино, Полоцкий р-н, Витебская обл.",NaN,55.623392,28.643457,Vitebsk,Polatsak
...,...,...,...,...,...,...,...,...,...,...,...
3469,Филиал № 6 ГУЗ «Гомельская центральная городск...,https://healthcare.by/instinfo.php?orgnum=4744,Гомель,Поликлиника,Polyclinic,"пр. Победы, 6, 246022, г. Гомель",http://https//gcgdp.by/ru/o-filiale-6/administ...,52.434359,31.005047,Gomel,Gomyel
3470,Филиал № 6 ГУЗ «Гомельская центральная городск...,https://healthcare.by/instinfo.php?orgnum=4673,Гомель,Поликлиника,Polyclinic,"ул. Новополесская, 11, 246003, г. Гомель",NaN,52.440964,30.999183,Gomel,Gomyel
3471,Филиал №1 ГУЗ «ВГПЦ» городская поликлиника №1,https://healthcare.by/instinfo.php?orgnum=26881,Витебск,Поликлиника,Polyclinic,"ул. Терешковой, 30, 210009, г. Витебск",NaN,55.193559,30.248739,Vitebsk,Vitebsk
3472,Центральная городская стоматологическая поликл...,https://healthcare.by/instinfo.php?orgnum=5555,Гродно,Поликлиника,Polyclinic,"ул. Суворова, 21, 230001, г. Гродно",http://cgsp.by,53.66036,23.812129,Grodno,Hrodna


In [ ]:
# df_gdf.to_file(
#     "healthcare_regions.gpkg",
#     driver="GPKG",
# )

In [ ]:
# df_gdf.to_csv("healthcare_regions.csv")

In [165]:
df[df['facility_title'].str.contains('Холомерь', na=False)]

,facility_title,link,settlement,facility_type,facility_type_eng,address,website,latitude,longitude,region,district
